In [1]:
# Import necessary libraries
import os
from vllm import LLM, SamplingParams
from vllm.steer_vectors.request import SteerVectorRequest, VectorConfig
from transformers import AutoTokenizer

# Set environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Initialize LLM with steering vector capability
llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    enable_steer_vector=True,
    enforce_eager=True,
    tensor_parallel_size=1
)

/media/volume/llm/miniconda3/envs/easysteer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-16 17:46:11 [utils.py:253] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'enable_steer_vector': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 11-16 17:46:11 [model.py:657] Resolved architecture: Qwen2ForCausalLM
INFO 11-16 17:46:11 [model.py:1746] Using max model len 131072


2025-11-16 17:46:12,059	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-16 17:46:12 [scheduler.py:211] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 11-16 17:46:12 [vllm.py:414] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:12 [core.py:94] Initializing a V1 LLM engine (v0.1.dev10891+ge8dee828a) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', enable_in

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.68it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.67it/s]
(EngineCore_DP0 pid=3569173) 


(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:15 [default_loader.py:314] Loading weights took 0.67 seconds
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:15 [steer_vector_model_runner_mixin.py:36] Initialized SteerVector worker manager
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:15 [steer_vector_model_runner_mixin.py:50] Wrapping model with steer vector support
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:15 [hidden_states_model_runner_mixin.py:90] Wrapped 28 decoder layers for hidden states capture
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:16 [gpu_model_runner.py:2971] Model loading took 3.3461 GiB and 1.179370 seconds
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:16 [gpu_worker.py:343] Available KV cache memory: 62.19 GiB
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:17 [kv_cache_utils.py:1247] GPU KV cache size: 2,328,768 tokens
(EngineCore_DP0 pid=3569173) INFO 11-16 17:46:17 [kv_cache_utils.py:1252] Maximum concurrency for 131,072 tokens per request: 17.77x
(EngineCore_DP0 p

# MATH500

In [2]:
import json
file_path = "/media/volume/llm/llm_steering_reasoning/data/MATH500/test.jsonl"

problems = []
answers = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["problem"])
        answers.append(item["answer"])

# 看看前两个
print("Problems:", problems[:2])
print("Answers:", answers[:2])


examples = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + prompt + "\nAssistant: <think>" for prompt in problems]


Problems: ['Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$']
Answers: ['\\left( 3, \\frac{\\pi}{2} \\right)', 'p - q']


In [3]:
example_answers = llm.generate(
    examples,
    SamplingParams(
        temperature=0,
        max_tokens=8192,
        skip_special_tokens=False,
    ),
)

Processed prompts: 100%|██████████| 500/500 [02:14<00:00,  3.71it/s, est. speed input: 348.83 toks/s, output: 14829.98 toks/s]


In [4]:
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
outputs = [output.outputs[0].text for output in example_answers]
extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
results = []
for i, llm_output in enumerate(outputs):
    gold = parse(f"${answers[i]}$", extraction_config=extraction_target)
    answer = parse(llm_output, extraction_config=extraction_target)
    result = verify(gold, answer)
    results.append(result)
accuracy = sum(results) / len(results)
print(accuracy)

0.722


In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
)
length = 0
for i in range(len(outputs)):
    length += len(tokenizer.tokenize(outputs[i], add_special_tokens=True))
print("Length: ", length/len(outputs))

Length:  4000.718


# GSM8k

In [6]:
import json
file_path = "/media/volume/llm/llm_steering_reasoning/data/gsm8k/test.jsonl"

problems = []
answers = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["question"])
        answers.append(item["answer"])

# 看看前两个
print("Problems:", problems[:2])
print("Answers:", answers[:2])


examples = ["Please reason step by step, and put your final answer within \\boxed{}.\nUser: " + prompt + "\nAssistant: <think>" for prompt in problems]


Problems: ["Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?']
Answers: ['Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18', 'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3']


In [7]:
example_answers = llm.generate(
    examples,
    SamplingParams(
        temperature=0,
        max_tokens=8192,
        skip_special_tokens=False,
    ),
)

Processed prompts: 100%|██████████| 1319/1319 [03:05<00:00,  7.12it/s, est. speed input: 598.66 toks/s, output: 16893.28 toks/s] 


In [8]:
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
outputs = [output.outputs[0].text for output in example_answers]
extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
results = []
for i, llm_output in enumerate(outputs):
    gold = parse(f"${answers[i]}$", extraction_config=extraction_target)
    answer = parse(llm_output, extraction_config=extraction_target)
    result = verify(gold, answer)
    results.append(result)
accuracy = sum(results) / len(results)
print(accuracy)

0.7952994692949203


In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
)
length = 0
for i in range(len(outputs)):
    length += len(tokenizer.tokenize(outputs[i], add_special_tokens=True))
print("Length: ", length/len(outputs))

Length:  2371.3563305534494


In [10]:
del llm